# Quantizing Qwen to INT4 (W4A16) with `llm-compressor`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxkimambo/inference-engineering-deep-dive/blob/main/docs/notebooks/quantization-qwen-w4a16.ipynb)

Companion notebook to **[§5.1 Quantization](https://inference.kimambo.de/chapters/techniques/quantization/)**.
It runs the whole post-training quantization (PTQ) pipeline end to end — load BF16 weights,
calibrate, quantize to **W4A16** (4-bit weights, 16-bit activations) with **GPTQ**, save the
compressed checkpoint, and sanity-check a generation.

Every step prints what it is doing and how long it took, so you can watch the pipeline work.

> **⚙️ Runtime.** This defaults to **Qwen2.5-0.5B-Instruct** so it fits and runs in a few minutes on
> a **free Colab T4 (16 GB)**. The chapter walks through the 7B model; to reproduce those exact size
> numbers, set `MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"` below and switch to a **Colab Pro L4 or A100**
> runtime — 7B GPTQ will OOM a T4.
>
> Enable the GPU first: **Runtime → Change runtime type → T4 GPU**.

## 0 · Confirm the GPU

Quantization needs a GPU. This should show a Tesla **T4** (free tier) — or an L4/A100 if you
upgraded the runtime for the 7B model.

In [ ]:
!nvidia-smi

## 1 · Install `llm-compressor`

We use [`llm-compressor`](https://github.com/vllm-project/llm-compressor), the vLLM-native
quantization library — its output loads straight into vLLM/SGLang.

> **⚠️ The version pin matters.** We pin `llmcompressor==0.8.0`. A floating install can pull a
> `transformers` / `compressed-tensors` / `pydantic` combination that mismatches, and the modifier
> config (a pydantic model) then fails to validate with a `ValidationError` at construction time.
> Pinning a known-good release is the fix.

In [ ]:
# Pinned to avoid the pydantic/version-mismatch ValidationError (see note above).
# Installs a compatible transformers, datasets, compressed-tensors, and pydantic.
!pip install -q llmcompressor==0.8.0

## 2 · Set up logging

A tiny helper so every step announces itself and reports its wall-clock time. `llm-compressor`
also logs its own progress (via `loguru`) during calibration — you'll see both streams.

In [ ]:
import logging, time, contextlib

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-5s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("quant")

@contextlib.contextmanager
def step(msg):
    """Log a step's start, then its elapsed time on completion."""
    log.info(f"\u25b6 {msg} \u2026")
    t0 = time.perf_counter()
    yield
    log.info(f"\u2713 {msg} \u2014 {time.perf_counter() - t0:,.1f}s")

## 3 · Pick the model and recipe knobs

**W4A16** = 4-bit weights, 16-bit activations. The activations stay at 16-bit on purpose: the
[sensitivity ladder](https://inference.kimambo.de/chapters/techniques/quantization/#what-the-sensitivity-ladder)
says weights tolerate quantization best, so we crush them and leave everything else alone.

- `GROUP_SIZE = 128` — weights share one scale factor per group of 128 (finer = better quality,
  more scales to store).
- `NUM_CALIBRATION_SAMPLES` — how many real samples GPTQ sees to measure value ranges. 256–512 is
  plenty; more is slower.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # swap to "Qwen/Qwen2.5-7B-Instruct" on an L4/A100
GROUP_SIZE = 128
NUM_CALIBRATION_SAMPLES = 256
MAX_SEQ_LEN = 2048

log.info(f"Model: {MODEL_ID}")
log.info(f"Recipe: W4A16, group_size={GROUP_SIZE}, calib_samples={NUM_CALIBRATION_SAMPLES}")

## 4 · Load the BF16 model

`dtype="auto"` loads the weights in their native precision (BF16 for Qwen). We also print the
parameter count and the in-memory weight size — this is the "before" number the quantization
will shrink.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

with step(f"Loading {MODEL_ID} in native precision"):
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype="auto")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

n_params = sum(p.numel() for p in model.parameters())
bytes_per = next(model.parameters()).element_size()
native_gb = n_params * bytes_per / 1e9

log.info(f"Parameters   : {n_params/1e9:.3f} B")
log.info(f"Native dtype : {next(model.parameters()).dtype} ({bytes_per} bytes/param)")
log.info(f"Weight size  : \u2248 {native_gb:.2f} GB (the 'before' number)")

## 5 · Inspect the model — layers and what they mean

Before touching the recipe, look at what you loaded. The module tree *is* the map you target:
`ignore=["lm_head"]` and `re:.*self_attn.*` refer to exactly these paths. `print(model)` dumps the
whole tree.

In [ ]:
print(model)

Reading the tree top to bottom (Qwen, like most decoder-only LLMs):

| Module path | What it is | Sensitivity |
|---|---|---|
| `model.embed_tokens` | token \u2192 vector lookup table | sensitive (leave in BF16) |
| `model.layers.{i}.self_attn.{q,k,v,o}_proj` | attention projections | the `k/v_proj` feed the KV cache |
| `model.layers.{i}.mlp.{gate,up,down}_proj` | the FFN \u2014 biggest, most robust | quantize hardest; `down_proj` least |
| `model.layers.{i}.*_layernorm` | RMSNorm \u2014 tiny | never quantized |
| `model.norm` | final norm | tiny |
| `lm_head` | vector \u2192 vocab logits (output head) | sensitive (leave in BF16) |

The `config` gives the shape numbers behind that tree:

In [ ]:
cfg = model.config
print("architecture      :", cfg.architectures[0])
print("hidden layers     :", cfg.num_hidden_layers)      # \u2192 the layers.{0..N-1} range for edge-layer regexes
print("hidden size       :", cfg.hidden_size)
print("intermediate size :", cfg.intermediate_size)      # the FFN width (gate/up/down_proj)
print("attention heads   :", cfg.num_attention_heads)
print("KV heads          :", cfg.num_key_value_heads)    # < attention heads \u21d2 grouped-query attention (GQA)
print("vocab size        :", cfg.vocab_size)

**Enumerate the Linear layers** — this is the recon you do before writing a recipe. Every name
printed here is a valid `targets`/`ignore` path; collapsing the layer index (`.0.` \u2192 `.N.`)
shows the handful of *families* you actually target.

In [ ]:
import re
import torch.nn as nn
from collections import Counter

linears = [(n, tuple(m.weight.shape)) for n, m in model.named_modules() if isinstance(m, nn.Linear)]
families = Counter(re.sub(r"\.\d+\.", ".N.", n) for n, _ in linears)

print(f"{len(linears)} Linear layers, in {len(families)} families:\n")
for name, count in families.items():
    example = next(shape for n, shape in linears if re.sub(r"\.\d+\.", ".N.", n) == name)
    print(f"{count:4d}  {name:34s} weight{example}")

**Where do the parameters actually live?** Quantization only pays off where the weight bytes
are. In a small model the embedding + head can be a surprising share; in a 7B the transformer
layers dominate — which is why weights-only W4A16 gets most of the win.

In [ ]:
from collections import defaultdict

buckets = defaultdict(int)
for name, p in model.named_parameters():
    if ".layers." in name:   key = f"transformer layers (\u00d7{cfg.num_hidden_layers})"
    elif "embed" in name:    key = "embed_tokens"
    elif name.startswith("lm_head"): key = "lm_head"
    else:                    key = "norms / other"
    buckets[key] += p.numel()

total = sum(buckets.values())
for key, n in sorted(buckets.items(), key=lambda kv: -kv[1]):
    print(f"{key:28s} {n/1e6:8.1f} M  ({100*n/total:4.1f}%)")

**Inspect a single weight tensor.** The gap between the *average* magnitude and the *absmax*
is the outlier problem quantization fights — one big value stretches the scale for the whole
group (the next step quantizes a real group by hand so you can watch it happen).

In [ ]:
w = model.model.layers[0].mlp.down_proj.weight
print("path   : model.layers.0.mlp.down_proj.weight")
print("shape  :", tuple(w.shape))
print("dtype  :", w.dtype)
print("device :", w.device)
print(f"min/max: {w.min().item():+.4f} / {w.max().item():+.4f}")
print(f"mean|w|: {w.abs().mean().item():.4f}")
print(f"absmax : {w.abs().max().item():.4f}   <- one outlier sets the whole group's scale")

## 6 · Peek under the hood: quantize 8 real weights by hand

Before running the real algorithm, let's reproduce the chapter's
[Step 1 math](https://inference.kimambo.de/chapters/techniques/quantization/#step-1-the-math-on-eight-real-weights)
on an **actual group of 8 weights** pulled from the model. This is plain round-to-nearest (RTN) —
the simplest possible scheme — so you can *see* the scale factor, the rounding, and the error
before GPTQ does it more cleverly across billions of weights.

In [ ]:
# One real group of 8 weights from the first layer's MLP down_proj.
group = model.model.layers[0].mlp.down_proj.weight.data[0, :8].float()

qmax = 7                                    # signed INT4 codes run [-7, 7]
absmax = group.abs().max()                  # the outlier sets the scale
scale = absmax / qmax                       # size of one quantization step
q = torch.clamp(torch.round(group / scale), -qmax, qmax)   # the stored INT4 codes
dequant = q * scale                         # what you get back
error = dequant - group

torch.set_printoptions(precision=4, sci_mode=False)
print("weights (BF16) :", group)
print("absmax         :", round(absmax.item(), 5))
print("scale S        :", round(scale.item(), 5))
print("quantized q    :", q.to(torch.int8).tolist())
print("dequantized w' :", dequant)
print("error (w'-w)   :", error)
print("max abs error  :", round(error.abs().max().item(), 5))

Notice the pattern from the chapter: the value that *set* the scale comes back exactly, while
small values near a big outlier round hardest — sometimes all the way to `0`. That's why **group
size matters**: a smaller group means an outlier inflates the scale for fewer neighbours. GPTQ
(next) improves on this by using loss-curvature information to compensate the not-yet-quantized
weights for each rounding error, minimizing the layer's *output* error rather than each weight's.

## 7 · Prepare calibration data

GPTQ needs a few hundred representative samples to measure real activation/weight ranges. We use
`ultrachat_200k` (general chat). **Use domain-matched data** — for a code model, calibrate on
code, not chat. We render each conversation through the chat template, then tokenize.

In [ ]:
from datasets import load_dataset

with step("Loading + preparing calibration data"):
    ds = load_dataset("HuggingFaceH4/ultrachat_200k", split=f"train_sft[:{NUM_CALIBRATION_SAMPLES}]")
    ds = ds.shuffle(seed=42)

    def to_text(example):
        return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}

    def tokenize(example):
        return tokenizer(example["text"], max_length=MAX_SEQ_LEN, truncation=True, add_special_tokens=False)

    ds = ds.map(to_text)
    ds = ds.map(tokenize, remove_columns=ds.column_names)

log.info(f"Calibration samples ready: {len(ds)}")

## 8 · Define the GPTQ recipe

`GPTQModifier` is the smart PTQ algorithm — **not** round-to-nearest.

- `targets="Linear"` — every `nn.Linear` is a candidate to quantize.
- `scheme="W4A16"` — 4-bit weights, 16-bit activations, `group_size=128`.
- `ignore=["lm_head"]` — keep the output head in BF16 (it's sensitive; sensitivity ladder).

This is the one control that matters most in practice. See **Targeting different layers** near the
end for how to protect edge layers, skip attention, or spare `down_proj`.

In [ ]:
from llmcompressor.modifiers.gptq import GPTQModifier

recipe = GPTQModifier(
    targets="Linear",       # quantize every nn.Linear\u2026
    scheme="W4A16",         # \u2026to 4-bit weights / 16-bit activations, group_size 128
    ignore=["lm_head"],     # \u2026except the output head (kept in BF16)
)
log.info(f"Recipe: GPTQ {recipe.scheme}, targets={recipe.targets}, ignore={recipe.ignore}")

## 9 · Calibrate + quantize (the actual work)

`oneshot()` runs the calibration data through the model, builds the GPTQ Hessians, and quantizes
each targeted layer **in place**. This is the slow step — a minute or two for 0.5B on a T4, ~20–60
min for 7B on an L4. Watch `llm-compressor`'s per-layer progress logs.

In [ ]:
from llmcompressor import oneshot

with step("Calibrating + quantizing (GPTQ one-shot)"):
    oneshot(
        model=model,
        dataset=ds,
        recipe=recipe,
        max_seq_length=MAX_SEQ_LEN,
        num_calibration_samples=NUM_CALIBRATION_SAMPLES,
    )

log.info("Model is now quantized in place.")

## 10 · Save the compressed checkpoint and measure the payoff

`save_compressed=True` writes the packed 4-bit weights. We then measure the on-disk size and
compare it to the BF16 "before" number from Step 4.

In [ ]:
import os

SAVE_DIR = MODEL_ID.split("/")[-1] + f"-W4A16-G{GROUP_SIZE}"

with step(f"Saving compressed checkpoint \u2192 {SAVE_DIR}"):
    model.save_pretrained(SAVE_DIR, save_compressed=True)
    tokenizer.save_pretrained(SAVE_DIR)

def dir_size_gb(path):
    total = sum(
        os.path.getsize(os.path.join(root, f))
        for root, _, files in os.walk(path) for f in files
    )
    return total / 1e9

quant_gb = dir_size_gb(SAVE_DIR)
print(f"BF16 weights (in memory) : {native_gb:6.2f} GB")
print(f"W4A16 checkpoint (disk)  : {quant_gb:6.2f} GB")
print(f"Shrink factor            : {native_gb / quant_gb:6.2f}x")

## 11 · Sanity-check a generation

The quantized model still generates. This is a smoke test, **not** a quality measurement — for
that, run perplexity / MMLU / your custom eval against the original BF16 weights (chapter §5.1.3).

In [ ]:
with step("Generating a sample completion from the quantized model"):
    messages = [{"role": "user", "content": "In one sentence, what does INT4 quantization trade away?"}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    out = model.generate(inputs, max_new_tokens=120, do_sample=False)

print(tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True))

## Targeting different layers

`targets` says *what to quantize*; `ignore` says *what to leave in BF16*. Both accept module
**class names**, exact **module paths**, or **regex** with a `re:` prefix. This is how you walk the
sensitivity ladder in practice. Print `model` to see Qwen's module paths. Recipes from least to
most conservative:

```python
# Default \u2014 quantize every Linear, keep only the output head
GPTQModifier(targets="Linear", scheme="W4A16", ignore=["lm_head"])

# Protect the edge layers \u2014 leave first 2 + last 2 transformer blocks in BF16
GPTQModifier(targets="Linear", scheme="W4A16",
             ignore=["lm_head", "re:model\\.layers\\.(0|1|22|23)\\..*"])

# MLP-only \u2014 skip all attention projections (most sensitive component)
GPTQModifier(targets="Linear", scheme="W4A16",
             ignore=["lm_head", "re:.*self_attn.*"])

# Spare a known-sensitive projection \u2014 down_proj often carries outliers
GPTQModifier(targets="Linear", scheme="W4A16",
             ignore=["lm_head", "re:.*down_proj"])
```

The layer indices in the edge-layer regex depend on the model: Qwen2.5-0.5B has 24 layers
(`0..23`), the 7B has 28 (`0..27`). Adjust to `(0|1|<n-2>|<n-1>)`.

> **🎯 The targeting workflow.** Start with **default**, run your evals. If quality regresses, don't
> abandon 4-bit — **add the regression's likely culprits to `ignore`** and re-run: edge layers, then
> attention, then `down_proj`. You're searching for the smallest set of BF16 exceptions that recovers
> quality.

> **💾 Memory tip for big models.** Add `sequential_targets=["Qwen2DecoderLayer"]` to the modifier to
> quantize **one decoder layer at a time**, keeping only that layer's activations in memory. Essential
> when the model barely fits — it's how the same recipe scales from 7B to 70B+.

## Use AWQ instead (one import)

AWQ often edges out GPTQ at 4-bit — it scales up salient weight channels (judged by activation
magnitude) before quantizing, so rounding hurts them less. Same `oneshot(...)` call, different
modifier:

```python
from llmcompressor.modifiers.awq import AWQModifier

recipe = AWQModifier(targets="Linear", scheme="W4A16", ignore=["lm_head"])
# \u2026identical oneshot(...) and save_pretrained(...)
```

## Colab tips & tricks

Practical things that save a session when working with models on Colab.

**Watch VRAM — it's the resource you'll run out of first.** Track allocated and peak usage; the
peak is what determines whether the next model fits.

In [ ]:
import torch

def gpu_mem(tag=""):
    alloc = torch.cuda.memory_allocated() / 1e9
    peak  = torch.cuda.max_memory_allocated() / 1e9
    print(f"{tag:22s} allocated={alloc:5.2f} GB   peak={peak:5.2f} GB")

gpu_mem("current")

**Free a model you're done with** before loading another — otherwise the old one still holds
VRAM and you OOM. `del` alone isn't enough; you need `gc.collect()` + `empty_cache()`:

```python
import gc, torch
del model                              # drop Python references first
gc.collect()
torch.cuda.empty_cache()               # return freed blocks to the driver
torch.cuda.reset_peak_memory_stats()   # so the next gpu_mem() peak is meaningful
```

If VRAM *stays* high after that, a stray reference is pinning it (a variable holding a tensor, an
output cell). The guaranteed reset is **Runtime → Restart session** — faster than hunting the leak.

**Gated / private models** (Llama, some Mistral) need a token — log in once per session:

```python
from huggingface_hub import notebook_login
notebook_login()          # paste a token from huggingface.co/settings/tokens
```

**Disk fills up too.** Every model downloads to `~/.cache/huggingface` (Colab gives ~100 GB, but a
few 7B checkpoints eat it). Check and clear:

```python
!df -h /                                    # free space
!du -sh ~/.cache/huggingface/hub/*          # what's cached
# !rm -rf ~/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct
```

**Persist your output** — Colab wipes local disk when the runtime recycles. Push the quantized
checkpoint somewhere durable *before* you close the tab: mount Drive
(`from google.colab import drive; drive.mount('/content/drive')`) and copy it there, `huggingface-cli
upload` it to a Hub repo, or `gcloud storage cp` to a bucket.

**`%pip` over `!pip`.** The `%` magic installs into the *running* kernel; a bare `!pip` can install
into a different environment and leave the import failing.

## Where to go from here

- **Persist it.** On a throwaway VM, push the checkpoint to durable storage:
  `gcloud storage cp -r ./<SAVE_DIR> gs://YOUR_BUCKET/models/` — then delete the GPU VM.
- **Serve it.** vLLM detects the quant format from the saved config, no special flags:
  `vllm serve ./<SAVE_DIR>`.
- **Prove it's good.** Compare perplexity + your custom eval against the original BF16 weights
  (chapter §5.1.3). Pick the most aggressive setting that still passes, and no further.

← Back to **[§5.1 Quantization](https://inference.kimambo.de/chapters/techniques/quantization/)**.